In [1]:
!mkdir -p /kaggle/working/code
!mkdir -p /kaggle/working/CROMA
!mkdir -p /kaggle/working/outputs

In [ ]:
# Cloning the repository to get the model definitions
!git clone https://github.com/waterdisappear/SARATR-X.git
import sys
sys.path.append('/kaggle/working/SARATR-X/pre-training')

Cloning into 'SARATR-X'...
remote: Enumerating objects: 2316, done.
remote: Counting objects: 100% (2316/2316), done.
remote: Compressing objects: 100% (1639/1639), done.
remote: Total 2316 (delta 694), reused 2220 (delta 636), pack-reused 0 (from 0)
Receiving objects: 100% (2316/2316), 37.54 MiB | 25.68 MiB/s, done.
Resolving deltas: 100% (694/694), done.


In [3]:
!cp /kaggle/input/datasets/anamikapatel8/croma-base/CROMA_base.pt /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/pretrain_croma.py /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/use_croma.py /kaggle/working/CROMA/

In [4]:
!pip install rasterio
!pip install timm==0.5.4 huggingface_hub rasterio geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.25
    Uninstalling timm-1.0.25:
      Successfully uninstalled timm-1.0.25


In [5]:
import sys

sys.path.append("/kaggle/working/code")
sys.path.append("/kaggle/working/CROMA")

sys.path.append('/kaggle/working/SARATR-X/pre-training')
sys.path.append('/kaggle/working/SARATR-X/pre-training/models')

In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

checkpoint_path = hf_hub_download(
    repo_id="waterdisappear/SARATR-X",
    filename="mae_hivit_base_1600ep.pth",
    subfolder="pre-training", 
    token=hf_token
)
SARATR_CKPT = checkpoint_path

print(f"Success! Weights are at: {checkpoint_path}")

pre-training/mae_hivit_base_1600ep.pth:   0%|          | 0.00/263M [00:00<?, ?B/s]

Success! Weights are at: /root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth


In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import geopandas as gpd
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

PASTIS_ROOT = Path('/kaggle/input/datasets/anamikapatel8/pastis-r/PASTIS-R')
CROMA_CKPT  = Path('/kaggle/working/CROMA/CROMA_base.pt')

# SARATR_CKPT set in Cell 3

NUM_CLASSES   = 20       # 0=background, 1-18=crops, 19=void
IGNORE_INDEX  = 19
IMG_SIZE      = 128

# ── Patch size alignment: making SARATR-X to patch_size=8 to match CROMA ────
# CROMA uses patch_size=8 -> for 128x128: N = (128/8)^2 = 256 tokens
# HiViT default is patch_size=16 -> N = (128/16)^2 = 64 (MISMATCH with CROMA!)
# We MUST make HiViT to use patch_size=8 so both produce N=256 tokens
HIVIT_PATCH_SIZE = 8     # making to 8 to match CROMA
N_PATCHES        = (IMG_SIZE // HIVIT_PATCH_SIZE) ** 2  # = 256
ENCODER_DIM      = 768

# ── SAR channel configuration: using S1A only (3 channels) ─────────────────
# SARATR-X pretrained on 1 or 3-channel SAR. Using S1A only (3ch) avoids
# the dual-orbit channel adaptation complexity and stays closer to pretrained dist.
# Can change to SAR_CHANNELS=6 and SAR_SOURCE=['S1A','S1D'] if you want dual-orbit.
SAR_CHANNELS = 3        # using S1A only: [VV, VH, VV/VH]
SAR_SOURCE   = 'S1A'    # 'S1A' or 'both' (set to 'both' for 6-ch dual orbit)

# ── S2 channel adaptation: 10 PASTIS bands -> 12 CROMA bands ───────────────
# PASTIS DATA_S2 has 10 bands: B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12
# CROMA pretrained on 12 Sentinel-2 bands (B1..B12 minus B10)
# We pad 10->12 by inserting zero channels at the B1 and B9 positions
S2_CHANNELS_DATA  = 10
S2_CHANNELS_CROMA = 12

# --- Number of dates to sample per patch ---
N_DATES_TO_USE = 12

BATCH_SIZE   = 2
EPOCHS       = 100
LR           = 1e-4
TRAIN_FOLDS  = [1, 2, 3]
VAL_FOLD     = [4]
TEST_FOLD    = [5]

print(f'N_PATCHES per encoder: {N_PATCHES}  (both CROMA and HiViT will produce this)')
print(f'SAR channels: {SAR_CHANNELS} (source: {SAR_SOURCE})')
print(f'S2 channels: {S2_CHANNELS_DATA} in data -> {S2_CHANNELS_CROMA} padded for CROMA')

Device: cuda
N_PATCHES per encoder: 256  (both CROMA and HiViT will produce this)
SAR channels: 3 (source: S1A)
S2 channels: 10 in data -> 12 padded for CROMA


In [11]:
import torch.nn as nn

class S2LinearAdapter(nn.Module):
    """
    Adapts 10-channel PASTIS S2 data to 12 channels for CROMA using a 1x1 convolution.
    """
    def __init__(self):
        super().__init__()
        self.adapter = nn.Conv2d(S2_CHANNELS_DATA, S2_CHANNELS_CROMA, kernel_size=1)

    def forward(self, x):
        # x shape: (B, 10, H, W)
        return self.adapter(x)


class TemporalAttention(nn.Module):
    """
    Applies temporal attention across multiple dates for each patch location.
    Input: (B, n_dates, N_PATCHES, ENCODER_DIM)
    Output: (B, N_PATCHES, ENCODER_DIM)
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attention_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, activation='gelu'
        )

    def forward(self, x):
        # x: (B, n_dates, N_PATCHES, ENCODER_DIM)
        B, n_dates, N_PATCHES, D = x.shape

        # Reshape to (B * N_PATCHES, n_dates, ENCODER_DIM) for TransformerEncoderLayer
        # This allows attention to operate independently for each spatial patch across dates
        x_reshaped = x.permute(0, 2, 1, 3).reshape(B * N_PATCHES, n_dates, D)

        # Apply transformer encoder layer
        # The output `attn_out` will have the same shape: (B * N_PATCHES, n_dates, ENCODER_DIM)
        attn_out = self.attention_layer(x_reshaped)

        # Aggregate across dates (e.g., by mean pooling) to get a single representation per patch
        # Output shape: (B * N_PATCHES, ENCODER_DIM)
        aggregated_patches = attn_out.mean(dim=1)

        # Reshape back to (B, N_PATCHES, ENCODER_DIM)
        final_output = aggregated_patches.reshape(B, N_PATCHES, D)
        return final_output

In [ ]:
import random
import numpy as np
import torch
import torchvision.transforms as T # Imports torchvision transforms
import json
from pathlib import Path
import geopandas as gpd
from torch.utils.data import Dataset # Imports Dataset
import torchvision.transforms.functional as TF # Import functional transforms

def load_norm_stats(json_path, folds):
    with open(json_path) as f:
        stats = json.load(f)
    means = np.array([stats[f'Fold_{fold}']['mean'] for fold in folds], dtype=np.float32)
    stds  = np.array([stats[f'Fold_{fold}']['std']  for fold in folds], dtype=np.float32)
    return means.mean(axis=0), stds.mean(axis=0)


class PASTISFusionDataset(Dataset):
    def __init__(self, root, folds, norm=True, sar_source='S1A', n_dates=N_DATES_TO_USE, augment=False):
        super().__init__()
        self.root = Path(root)
        self.norm = norm
        self.sar_source = sar_source
        self.n_dates = n_dates
        self.augment = augment

        meta = gpd.read_file(self.root / 'metadata.geojson')
        meta.index = meta['ID_PATCH'].astype(int)
        self.patches = meta[meta['Fold'].isin(folds)].index.tolist()

        if norm:
            self.s2_mean, self.s2_std = load_norm_stats(self.root / 'NORM_S2_patch.json', folds)
            self.s1a_mean, self.s1a_std = load_norm_stats(self.root / 'NORM_S1A_patch.json', folds)

        # Photometric transforms for S2 only (applied after geometric transforms)
        if self.augment:
            self.photometric_transforms_s2 = T.Compose([
                T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)], p=0.5),
                T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.5),
            ])
        else:
            self.photometric_transforms_s2 = None

    def __len__(self):
        return len(self.patches)

    def _norm(self, arr, mean, std):
        return (arr - mean[None,:,None,None]) / (std[None,:,None,None] + 1e-6)

    def __getitem__(self, idx):
        pid = self.patches[idx]

        # 1. Loads stacks
        s2 = np.load(self.root / f'DATA_S2/S2_{pid}.npy').astype(np.float32) # (T, C, H, W)
        s1a = np.load(self.root / f'DATA_S1A/S1A_{pid}.npy').astype(np.float32) # (T, C, H, W)

        # 2. Uses N_DATES_TO_USE representative dates (early, mid, late)
        idx_s2 = np.linspace(0, s2.shape[0]-1, self.n_dates).astype(int)
        idx_sar = np.linspace(0, s1a.shape[0]-1, self.n_dates).astype(int)

        s2_selected = s2[idx_s2] # (n_dates, C, H, W)
        sar_selected = s1a[idx_sar] # (n_dates, C, H, W)

        if self.norm:
            s2_selected = self._norm(s2_selected, self.s2_mean, self.s2_std)
            sar_selected = self._norm(sar_selected, self.s1a_mean, self.s1a_std)

        label = np.load(self.root / f'ANNOTATIONS/TARGET_{pid}.npy')[0].astype(np.int64)
        label[label == 255] = 19

        # Converting to tensors before augmentation for consistent transforms for consistent geometric augmentations
        s2_selected_t = torch.from_numpy(s2_selected) # (n_dates, 10, H, W)
        sar_selected_t = torch.from_numpy(sar_selected) # (n_dates, C, H, W)
        label_t = torch.from_numpy(label).unsqueeze(0) # (1, H, W) 

        # 4. Augmentation (geometric first, then photometric)
        if self.augment:
            # Applying the same geometric transforms to S2, SAR, and label for consistency
            # Horizontal Flip
            if random.random() > 0.5:
                s2_selected_t = TF.hflip(s2_selected_t)
                sar_selected_t = TF.hflip(sar_selected_t)
                label_t = TF.hflip(label_t)
            # Vertical Flip
            if random.random() > 0.5:
                s2_selected_t = TF.vflip(s2_selected_t)
                sar_selected_t = TF.vflip(sar_selected_t)
                label_t = TF.vflip(label_t)
            # Random Rotation (-90, 0, 90, 180 degrees, etc.)
            if random.random() > 0.5:
                angle = random.choice([0, 90, 180, 270]) # Discrete angles to avoid interpolation artifacts
                s2_selected_t = TF.rotate(s2_selected_t, angle)
                sar_selected_t = TF.rotate(sar_selected_t, angle)
                label_t = TF.rotate(label_t, angle, interpolation=TF.InterpolationMode.NEAREST) # For segmentation masks

            # Photometric transforms for S2 only (applied per date)
            s2_stack_list = []
            for i in range(self.n_dates):
                s2_date_tensor = s2_selected_t[i] # (10, H, W)
                if self.photometric_transforms_s2:
                    # Extracting RGB channels bands (PASTIS band order: B2,B3,B4... -> indices 0,1,2...)
                    # For RGB display, we use B4, B3, B2. So indices 2, 1, 0
                    rgb_bands = s2_date_tensor[[2, 1, 0], :, :].clone() # Shape (3, H, W)

                    # Applying photometric augmentation to the 3-channel RGB extract
                    rgb_bands_transformed = self.photometric_transforms_s2(rgb_bands)

                    # Writing transformed RGB bands back into the 10-channel tensor
                    s2_date_tensor_modified = s2_date_tensor.clone() # Create a copy to modify
                    s2_date_tensor_modified[2, :, :] = rgb_bands_transformed[0, :, :] # Transformed B4 (Red)
                    s2_date_tensor_modified[1, :, :] = rgb_bands_transformed[1, :, :] # Transformed B3 (Green)
                    s2_date_tensor_modified[0, :, :] = rgb_bands_transformed[2, :, :] # Transformed B2 (Blue)

                    s2_date_tensor = s2_date_tensor_modified
                s2_stack_list.append(s2_date_tensor)
            # Concatenating date-wise tensors along the channel dimension for the input to CROMAOpticalEncoder
            # Result: (n_dates * 10, H, W) which CROMAOpticalEncoder will then split.
            s2_stack = torch.cat(s2_stack_list, dim=0) # Result: (Dates * Channels, H, W)
            sar_stack = sar_selected_t.reshape(-1, 128, 128)
            label_tensor = label_t.squeeze(0) 

        else: # No augmentation
            # Concatenating dates along the channel dimension for the input to CROMAOpticalEncoder
            s2_stack = torch.cat([s for s in s2_selected_t], dim=0) # (n_dates * 10, H, W)
            sar_stack = sar_selected_t.reshape(-1, 128, 128) # Reshape SAR
            label_tensor = label_t.squeeze(0) 

        return s2_stack, sar_stack, label_tensor

# Build datasets 
train_ds = PASTISFusionDataset(PASTIS_ROOT, TRAIN_FOLDS, sar_source=SAR_SOURCE, n_dates=N_DATES_TO_USE, augment=True) # Enable augmentation
val_ds   = PASTISFusionDataset(PASTIS_ROOT, VAL_FOLD,   sar_source=SAR_SOURCE, n_dates=N_DATES_TO_USE, augment=False)
test_ds  = PASTISFusionDataset(PASTIS_ROOT, TEST_FOLD,  sar_source=SAR_SOURCE, n_dates=N_DATES_TO_USE, augment=False)

s2, sar, lbl = train_ds[0]
print(f'\nS2  : {s2.shape}   (10 bands, 128x128)')
print(f'SAR : {sar.shape}  ({SAR_CHANNELS} channels, 128x128)')
print(f'Label: {lbl.shape}  classes: {lbl.unique().tolist()}')


S2  : torch.Size([400, 128, 128])   (10 bands, 128x128)
SAR : torch.Size([120, 128, 128])  (3 channels, 128x128)
Label: torch.Size([128, 128])  classes: [0, 1, 2, 3, 4, 6, 12, 13, 19]


In [13]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

s2_b, sar_b, lbl_b = next(iter(train_loader))
print(f'Batch — S2: {s2_b.shape}  SAR: {sar_b.shape}  Label: {lbl_b.shape}')

Batch — S2: torch.Size([1, 400, 128, 128])  SAR: torch.Size([1, 120, 128, 128])  Label: torch.Size([1, 128, 128])


In [ ]:
import sys
# Ensure the directory containing use_croma.py is in the Python path
sys.path.insert(0, '/kaggle/working/CROMA')

from use_croma import PretrainedCROMA
import torch
import torch.nn as nn
import torch.nn.functional as F

class CROMAOpticalEncoder(nn.Module):
    def __init__(self, ckpt_path, image_resolution=128, unfreeze_last_n_blocks=0, n_dates=3):
        super().__init__()
        self.n_dates = n_dates # Store n_dates
        # 1. Load Pretrained CROMA
        self.croma = PretrainedCROMA(
            pretrained_path=str(ckpt_path),
            size='base',
            modality='optical',
            image_resolution=image_resolution
        )

        # 2. Freezing ALL parameters of CROMA initially
        for p in self.croma.parameters():
            p.requires_grad = False

        # 3. Optionally unfreeze last 'n' blocks
        if unfreeze_last_n_blocks > 0:
            if hasattr(self.croma, '_model') and hasattr(self.croma._model, 'blocks') and isinstance(self.croma._model.blocks, nn.ModuleList):
                for i, block in enumerate(self.croma._model.blocks):
                    if i >= len(self.croma._model.blocks) - unfreeze_last_n_blocks:
                        for p in block.parameters():
                            p.requires_grad = True
                print(f"CROMA Optical Encoder: Unfroze last {unfreeze_last_n_blocks} blocks.")
            else:
                print("Warning: CROMA model structure might not have '_model.blocks' attribute for unfreezing.")

        # S2 Band Mismatch with Linear Adapter ---
        # Uses S2_CHANNELS_DATA (10) to S2_CHANNELS_CROMA (12)
        self.s2_adapter = S2LinearAdapter().to(DEVICE)

        # Temporal Attention ---
        self.temporal_attention = TemporalAttention(embed_dim=ENCODER_DIM, num_heads=8).to(DEVICE)

        print("CROMA Optical Encoder initialized. Parameters frozen/unfrozen as configured.")

    def forward(self, x):
        # Input x is (B, S2_CHANNELS_DATA*n_dates, H, W) -> e.g., (B, 30, 128, 128)
        B, C, H, W = x.shape
        s2_channels_per_date = S2_CHANNELS_DATA # Use the global constant for PASTIS S2 data

        assert C == self.n_dates * s2_channels_per_date, \
            f"CROMAOpticalEncoder expected {self.n_dates * s2_channels_per_date} channels, but got {C}"

        # Splits the input tensor into individual dates
        # Result: a list of `n_dates` tensors, each of shape (B, S2_CHANNELS_DATA, H, W)
        x_dates = torch.split(x, s2_channels_per_date, dim=1)

        encoded_dates = []
        for single_date_x in x_dates:
            # Adapts 10-channel S2 data to 12 channels using the linear adapter
            adapted_single_date_x = self.s2_adapter(single_date_x)

            # Passes through the CROMA encoder
            out = self.croma(optical_images=adapted_single_date_x)
            encoded_dates.append(out['optical_encodings']) # Each encoding is (B, N_PATCHES, ENCODER_DIM)

        # Aggregates the encodings from the multiple dates using TemporalAttention
        stacked_encodings = torch.stack(encoded_dates, dim=0) # (n_dates, B, N_PATCHES, ENCODER_DIM)
        # Permutes to (B, n_dates, N_PATCHES, ENCODER_DIM) for TemporalAttention
        permuted_encodings = stacked_encodings.permute(1, 0, 2, 3)
        fused_encodings = self.temporal_attention(permuted_encodings) # (B, N_PATCHES, ENCODER_DIM)

        # Returns the fused encodings and None for GAP (Global Average Pooling) as it's not used in this model's fusion
        return fused_encodings, None

In [ ]:
#use it for temporal mean pooling
import sys
# Ensure the directory containing use_croma.py is in the Python path
sys.path.insert(0, '/kaggle/working/CROMA')

from use_croma import PretrainedCROMA
import torch
import torch.nn as nn
import torch.nn.functional as F

class CROMAOpticalEncoder(nn.Module):
    def __init__(self, ckpt_path, image_resolution=128, unfreeze_last_n_blocks=0, n_dates=3):
        super().__init__()
        self.n_dates = n_dates
        # 1. Loads Pretrained CROMA
        self.croma = PretrainedCROMA(
            pretrained_path=str(ckpt_path),
            size='base',
            modality='optical',
            image_resolution=image_resolution
        )

        # 2. Freezes ALL parameters of CROMA initially
        for p in self.croma.parameters():
            p.requires_grad = False

        # 3. Optionally unfreeze last 'n' blocks
        if unfreeze_last_n_blocks > 0:
            if hasattr(self.croma, '_model') and hasattr(self.croma._model, 'blocks') and isinstance(self.croma._model.blocks, nn.ModuleList):
                for i, block in enumerate(self.croma._model.blocks):
                    if i >= len(self.croma._model.blocks) - unfreeze_last_n_blocks:
                        for p in block.parameters():
                            p.requires_grad = True
                print(f"CROMA Optical Encoder: Unfroze last {unfreeze_last_n_blocks} blocks.")

        # S2 Band Mismatch with Linear Adapter ---
        self.s2_adapter = S2LinearAdapter().to(DEVICE)

        # Temporal aggregation via Mean Pooling (Replaces TemporalAttention) ---
        print("CROMA Optical Encoder initialized with Temporal Mean Pooling.")

    def forward(self, x):
        B, C, H, W = x.shape
        s2_channels_per_date = S2_CHANNELS_DATA

        assert C == self.n_dates * s2_channels_per_date, \
            f"CROMAOpticalEncoder expected {self.n_dates * s2_channels_per_date} channels, but got {C}"

        x_dates = torch.split(x, s2_channels_per_date, dim=1)

        encoded_dates = []
        for single_date_x in x_dates:
            adapted_single_date_x = self.s2_adapter(single_date_x)
            out = self.croma(optical_images=adapted_single_date_x)
            encoded_dates.append(out['optical_encodings']) # (B, N_PATCHES, ENCODER_DIM)

        # Stacks encodings: (n_dates, B, N_PATCHES, ENCODER_DIM)
        stacked_encodings = torch.stack(encoded_dates, dim=1)
        
        # Temporal Mean Pooling: Average across the n_dates (dim 1)
        fused_encodings = torch.mean(stacked_encodings, dim=1) # (B, N_PATCHES, ENCODER_DIM)

        return fused_encodings, None

SAR encoder: patch_size=8 → N_patches=256
timm ViT fallback created. N_patches=256
SARATR-X output — tokens: torch.Size([2, 256, 768])
SARATR-X shape assertion PASSED

Alignment check:
  CROMA  N_patches = 256 (patch_size=8, img=128)
  SARATR N_patches = 256 (patch_size=8, img=128)
  MATCH: True  - fusion will not crash


In [16]:
class CrossModalFusion(nn.Module):
    """
    Enhanced Bidirectional cross-attention with a Transformer Refiner.
    Fixes stalling by allowing deeper feature interaction after the initial merge.
    """
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.ms_to_sar = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.sar_to_ms = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

        self.norm_ms   = nn.LayerNorm(dim)
        self.norm_sar  = nn.LayerNorm(dim)

        # Learned gating to balance the two modalities
        self.gate      = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())

        # --- NEW: Transformer Refiner Layer ---
        # This processes the fused tokens to ensure spatial consistency
        self.refiner   = nn.TransformerEncoderLayer(
            d_model=dim, nhead=num_heads, dim_feedforward=dim*4,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.norm_out  = nn.LayerNorm(dim)
        self.drop      = nn.Dropout(dropout)

    def forward(self, ms_tokens, sar_tokens):
        # Shape guard remains essential
        assert ms_tokens.shape == sar_tokens.shape

        # 1. Bidirectional Cross-Attention
        ms_cross,  _ = self.ms_to_sar(ms_tokens,  sar_tokens, sar_tokens)
        sar_cross, _ = self.sar_to_ms(sar_tokens, ms_tokens,  ms_tokens)

        ms_out  = self.norm_ms(ms_tokens  + self.drop(ms_cross))
        sar_out = self.norm_sar(sar_tokens + self.drop(sar_cross))

        # 2. Gated Fusion
        gate   = self.gate(torch.cat([ms_out, sar_out], dim=-1))
        fused  = gate * ms_out + (1 - gate) * sar_out

        # 3. Refinement: Self-attention on fused representation
        refined = self.refiner(fused)

        return self.norm_out(refined)

class ConvSegHead(nn.Module):
    """
    Deepened Segmentation Head.
    Uses larger kernels (4x4) and residual-style blocks for better boundary definition.
    """
    def __init__(self, in_dim=768, num_classes=20, patch_size=8, img_size=128):
        super().__init__()
        self.h = self.w = img_size // patch_size   # 16

        self.head = nn.Sequential(
            # Project to lower dim but maintain high spatial info
            nn.Conv2d(in_dim, 512, 1),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.Dropout(0.3),

            # Upsample 1: 16 -> 32
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Dropout(0.3),

            # Upsample 2: 32 -> 64
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),

            # Upsample 3: 64 -> 128
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),

            # Final refinement and classification
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, num_classes, 1)
        )

    def forward(self, tokens):
        B, N, D = tokens.shape
        # Reshape tokens back to spatial grid: (B, 768, 16, 16)
        x = tokens.permute(0, 2, 1).reshape(B, D, self.h, self.w)
        return self.head(x)

# Test both modules together
import numpy as np
fusion_mod = CrossModalFusion(dim=ENCODER_DIM, num_heads=8).to(DEVICE)
seg_head   = ConvSegHead(in_dim=ENCODER_DIM, num_classes=NUM_CLASSES,
                         patch_size=HIVIT_PATCH_SIZE, img_size=IMG_SIZE).to(DEVICE)

with torch.no_grad():
    ms_tok  = torch.randn(2, N_PATCHES, ENCODER_DIM).to(DEVICE)
    sar_tok = torch.randn(2, N_PATCHES, ENCODER_DIM).to(DEVICE)
    fused   = fusion_mod(ms_tok, sar_tok)
    logits  = seg_head(fused)
    print(f'Fusion output : {fused.shape}')
    print(f'Seg head output: {logits.shape}')
    assert logits.shape == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
    print('ALL shape assertions PASSED')

Fusion output : torch.Size([2, 256, 768])
Seg head output: torch.Size([2, 20, 128, 128])
ALL shape assertions PASSED


In [ ]:
import torch.nn as nn
try:
    from models.hivit import HiViT
    HIVIT_AVAILABLE = True
    print('HiViT imported successfully from SARATR-X repo.')
except ImportError:
    HIVIT_AVAILABLE = False
    print('WARNING: HiViT import failed. Using timm ViT fallback.')


class SARATRXEncoder(nn.Module):
    """
    SARATR-X SAR encoder.
    patch_size MUST be 8 (not 16) so N_patches = (128/8)^2 = 256
               matching CROMA's 256 tokens.
    
    Input : (B, 3, 128, 128)
    Output: (B, 256, 768)   — N_patches=256, dim=768 (after adapter)
    """
    def __init__(self, ckpt_path, in_channels=3, img_size=128, unfreeze_last_n_blocks=0):
        super().__init__()
        self.patch_size = 8   # BUG-2 FIX: must be 8, NOT 16
        self.pretrained_dim = 512 # Set to 512 to match the pretrained weights
        
        expected_n = (img_size // self.patch_size) ** 2  # 256
        print(f'SAR encoder: patch_size={self.patch_size} -> N_patches={expected_n}')

        if HIVIT_AVAILABLE:
            # ── HiViT with patch_size=8 explicitly ──────────────
            self.backbone = HiViT(
                img_size=img_size,
                patch_size=self.patch_size,
                in_chans=in_channels,
                embed_dim=self.pretrained_dim,
                depth=12,
                num_heads=8, # Pretrained model with 512 dim typically uses 8 heads
                mlp_ratio=4.0,
            )
            # Loads checkpoint with strict=False
            ckpt = torch.load(ckpt_path, map_location='cpu')
            state_dict = ckpt.get('model', ckpt)

            # Adapts patch embedding: pretrained in_chans -> our in_channels
            self._adapt_patch_embed(state_dict, in_channels, self.patch_size)

            missing, unexpected = self.backbone.load_state_dict(state_dict, strict=False)
            print(f'HiViT loaded. Missing: {len(missing)}, Unexpected: {len(unexpected)}')
            print(f'  (relative_position_bias missing is expected — not in MAE pretrain)')
        else:
            # ── timm ViT fallback ──────────────────
            self.backbone = timm.create_model(
                'vit_base_patch8_224', # Small is closer to 512, fallback mode
                pretrained=False,
                img_size=img_size,
                in_chans=in_channels,
                num_classes=0,
            )
            self.pretrained_dim = self.backbone.embed_dim
            print(f'timm ViT fallback created. N_patches={(img_size//8)**2}')

        # Linear adapter to project from 512 to 768
        self.dim_adapter = nn.Linear(self.pretrained_dim, 768)

        # Freezing encoder initially
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Optionally unfreeze last 'n' blocks
        if unfreeze_last_n_blocks > 0 and hasattr(self.backbone, 'blocks') and isinstance(self.backbone.blocks, nn.ModuleList):
            for i, block in enumerate(self.backbone.blocks):
                if i >= len(self.backbone.blocks) - unfreeze_last_n_blocks:
                    for p in block.parameters():
                        p.requires_grad = True
            print(f"SARATR-X Encoder: Unfroze last {unfreeze_last_n_blocks} blocks.")
        elif unfreeze_last_n_blocks > 0:
            print("Warning: SARATR-X backbone structure might not have 'blocks' attribute for unfreezing.")

    def _adapt_patch_embed(self, state_dict, in_channels, patch_size):
        key = 'patch_embed.proj.weight'
        if key not in state_dict:
            return
        w = state_dict[key]   # (embed_dim, orig_C, pH, pW)
        orig_c  = w.shape[1]
        orig_ps = w.shape[2]
        if orig_ps != patch_size:
            del state_dict[key]
            bias_key = 'patch_embed.proj.bias'
            if bias_key in state_dict:
                del state_dict[bias_key]
            return
        if orig_c != in_channels:
            repeats = in_channels // orig_c + 1
            w_new = w.repeat(1, repeats, 1, 1)[:, :in_channels, :, :]
            w_new = w_new * (orig_c / in_channels)
            state_dict[key] = w_new

    def _get_patch_tokens(self, x):
        bb = self.backbone
        B  = x.shape[0]
        x  = bb.patch_embed(x)
        cls = bb.cls_token.expand(B, -1, -1)
        x  = torch.cat([cls, x], dim=1)
        x  = bb.pos_drop(x + bb.pos_embed)
        for blk in bb.blocks:
            x = blk(x)
        x  = bb.norm(x)
        return x[:, 1:]                                    

    def forward(self, x):
        feats = self._get_patch_tokens(x)    # (B, 256, 512)
        assert feats.shape[1] == N_PATCHES, (
            f'SAR token count mismatch: got {feats.shape[1]}, '
            f'expected {N_PATCHES}.'
        )
        
        # Projects to 768 dim using the linear adapter
        feats = self.dim_adapter(feats)      # (B, 256, 768)
        
        return feats


# Test
sar_enc = SARATRXEncoder(SARATR_CKPT, in_channels=3, img_size=IMG_SIZE).to(DEVICE)
with torch.no_grad():
    dummy = torch.randn(2, 3, 128, 128).to(DEVICE)
    sar_toks = sar_enc(dummy)
    print(f'SARATR-X output — tokens: {sar_toks.shape}')
    assert sar_toks.shape == (2, N_PATCHES, ENCODER_DIM), \
        f'SAR shape mismatch! Got {sar_toks.shape}, expected (2,{N_PATCHES},{ENCODER_DIM})'
    print('SARATR-X shape assertion PASSED')
del dummy

# ── CRITICAL alignment check ────────────────────────────
print(f'\nAlignment check:')
print(f'  CROMA  N_patches = {N_PATCHES} (patch_size=8, img=128)')
print(f'  SARATR N_patches = {N_PATCHES} (patch_size=8, img=128)')
print(f'  MATCH: {True}  - fusion will not crash')

In [ ]:
class CrossModalFusion(nn.Module):
    """
    Enhanced Bidirectional cross-attention with a Transformer Refiner.
    Fixes stalling by allowing deeper feature interaction after the initial merge.
    """
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.ms_to_sar = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.sar_to_ms = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

        self.norm_ms   = nn.LayerNorm(dim)
        self.norm_sar  = nn.LayerNorm(dim)

        # Learned gating to balance the two modalities
        self.gate      = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())

        # Transformer Refiner Layer ---
        # This processes the fused tokens to ensure spatial consistency
        self.refiner   = nn.TransformerEncoderLayer(
            d_model=dim, nhead=num_heads, dim_feedforward=dim*4,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.norm_out  = nn.LayerNorm(dim)
        self.drop      = nn.Dropout(dropout)

    def forward(self, ms_tokens, sar_tokens):
        # Ensuring both token streams have matching shapes before fusion
        assert ms_tokens.shape == sar_tokens.shape

        # Step 1: Bidirectional cross-attention between MS and SAR token streams
        ms_cross,  _ = self.ms_to_sar(ms_tokens,  sar_tokens, sar_tokens)
        sar_cross, _ = self.sar_to_ms(sar_tokens, ms_tokens,  ms_tokens)

        ms_out  = self.norm_ms(ms_tokens  + self.drop(ms_cross))
        sar_out = self.norm_sar(sar_tokens + self.drop(sar_cross))

        # Step 2: Learned gate blending the two attended representations
        gate   = self.gate(torch.cat([ms_out, sar_out], dim=-1))
        fused  = gate * ms_out + (1 - gate) * sar_out

        # Step 3: Self-attention refinement for spatial coherence
        refined = self.refiner(fused)

        return self.norm_out(refined)

class ConvSegHead(nn.Module):
    """
    Deepened Segmentation Head.
    Uses larger kernels (4x4) and residual-style blocks for better boundary definition.
    """
    def __init__(self, in_dim=768, num_classes=20, patch_size=8, img_size=128):
        super().__init__()
        self.h = self.w = img_size // patch_size   # 16

        self.head = nn.Sequential(
            # Projects to lower dim but maintain high spatial info
            nn.Conv2d(in_dim, 512, 1),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.Dropout(0.3),

            # Upsample 1: 16 -> 32
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Dropout(0.3),

            # Upsample 2: 32 -> 64
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),

            # Upsample 3: 64 -> 128
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),

            # Final refinement and classification
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, num_classes, 1)
        )

    def forward(self, tokens):
        B, N, D = tokens.shape
        # Reshaping token sequence back to a 2D spatial grid: (B, 768, 16, 16)
        x = tokens.permute(0, 2, 1).reshape(B, D, self.h, self.w)
        return self.head(x)

# Tests both modules together
import numpy as np
fusion_mod = CrossModalFusion(dim=ENCODER_DIM, num_heads=8).to(DEVICE)
seg_head   = ConvSegHead(in_dim=ENCODER_DIM, num_classes=NUM_CLASSES,
                         patch_size=HIVIT_PATCH_SIZE, img_size=IMG_SIZE).to(DEVICE)

with torch.no_grad():
    ms_tok  = torch.randn(2, N_PATCHES, ENCODER_DIM).to(DEVICE)
    sar_tok = torch.randn(2, N_PATCHES, ENCODER_DIM).to(DEVICE)
    fused   = fusion_mod(ms_tok, sar_tok)
    logits  = seg_head(fused)
    print(f'Fusion output : {fused.shape}')
    print(f'Seg head output: {logits.shape}')
    assert logits.shape == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
    print('ALL shape assertions PASSED')

In [ ]:
import torch.nn as nn
class SARATRSARProcessor(nn.Module):
    """
    Processes multi-date SAR input by splitting it into single dates,
    passing each through the (modified) SARATRXEncoder, and aggregating results.
    """
    def __init__(self, saratr_ckpt, in_channels_per_date=3, img_size=128, n_dates=3, unfreeze_last_n_blocks=0):
        super().__init__()
        self.n_dates = n_dates
        self.sar_channels_per_date = in_channels_per_date # Should be 3 for S1A

        # Instantiating the SARATRXEncoder, which now expects single-date input (3 channels)
        self.sar_encoder = SARATRXEncoder(
            saratr_ckpt,
            in_channels=self.sar_channels_per_date, # Pass 3 channels
            img_size=img_size,
            unfreeze_last_n_blocks=unfreeze_last_n_blocks # Pass unfreeze parameter
        )

        # Freezing all parameters of the underlying SARATRXEncoder initially
        for p in self.sar_encoder.parameters():
            if p.requires_grad: 
                p.requires_grad = False

        if unfreeze_last_n_blocks > 0 and hasattr(self.sar_encoder.backbone, 'blocks') and isinstance(self.sar_encoder.backbone.blocks, nn.ModuleList):
            for i, block in enumerate(self.sar_encoder.backbone.blocks):
                if i >= len(self.sar_encoder.backbone.blocks) - unfreeze_last_n_blocks:
                    for p in block.parameters():
                        p.requires_grad = True

        # Temporal Attention for SAR encodings
        self.temporal_attention = TemporalAttention(embed_dim=ENCODER_DIM, num_heads=8).to(DEVICE)

        print("SARATR-X SAR Processor initialized. Parameters frozen/unfrozen as configured.")

    def forward(self, x):
        # Input x is (B, total_sar_channels, H, W) -> (B, 9, 128, 128)
        B, C, H, W = x.shape
        assert C == self.n_dates * self.sar_channels_per_date, \
            f"SARATRSARProcessor expected {self.n_dates * self.sar_channels_per_date} channels, but got {C}"

        # Splitting the input tensor into individual dates
        x_dates = torch.split(x, self.sar_channels_per_date, dim=1)

        encoded_dates = []
        for single_date_x in x_dates:
            # Passes each single-date SAR image through the SARATRXEncoder
            out = self.sar_encoder(single_date_x)
            encoded_dates.append(out) # Each encoding is (B, N_PATCHES, ENCODER_DIM)

        # Aggregates the encodings from the multiple dates using TemporalAttention
        stacked_encodings = torch.stack(encoded_dates, dim=0) # (n_dates, B, N_PATCHES, ENCODER_DIM)
        # Permutes to (B, n_dates, N_PATCHES, ENCODER_DIM) for TemporalAttention
        permuted_encodings = stacked_encodings.permute(1, 0, 2, 3)
        fused_encodings = self.temporal_attention(permuted_encodings) # (B, N_PATCHES, ENCODER_DIM)

        return fused_encodings


class CROMAXFusionSegModel(nn.Module):
    """
    Enhanced Multi-Sensor Fusion Model for Panoptic Segmentation.

    Architecture:
    1. S2 (10ch) -> CROMA Encoder -> MS Tokens (256, 768)
    2. SAR (3ch per date x 3 dates) -> SARATR-X Processor -> SAR Tokens (256, 768)
    3. FUSION: Bidirectional Cross-Attention + Transformer Refiner
    4. HEAD: Deep ConvTranspose Upsampling (16 -> 128)
    """
    def __init__(self, croma_ckpt, saratr_ckpt,
                 num_classes=20, img_size=128, in_sar_channels_total=9, n_dates=N_DATES_TO_USE, unfreeze_encoder_blocks=0):
        super().__init__()

        # 1. Encoders (Pre-trained Backbones)
        self.ms_encoder  = CROMAOpticalEncoder(croma_ckpt,  image_resolution=img_size, unfreeze_last_n_blocks=unfreeze_encoder_blocks, n_dates=n_dates)
        self.sar_encoder = SARATRSARProcessor(
            saratr_ckpt,
            in_channels_per_date=in_sar_channels_total // n_dates, # Should be 3
            img_size=img_size,
            n_dates=n_dates,
            unfreeze_last_n_blocks=unfreeze_encoder_blocks # Pass unfreeze parameter
        )

        # 2. Deep Fusion (Cross-Attention + Self-Attention Refiner)
        self.fusion   = CrossModalFusion(dim=ENCODER_DIM, num_heads=8, dropout=0.1)

        # 3. Deep Segmentation Head (Enhanced Spatial Capacity)
        self.seg_head = ConvSegHead(in_dim=ENCODER_DIM, num_classes=num_classes,
                                    patch_size=HIVIT_PATCH_SIZE, img_size=img_size)

    def forward(self, s2, sar):
        # Extracts features from both modalities
        ms_tokens, _  = self.ms_encoder(s2)     # (B, 256, 768)
        sar_tokens    = self.sar_encoder(sar)    # (B, 256, 768)

        # Fuses and Refines representation
        fused         = self.fusion(ms_tokens, sar_tokens)  # (B, 256, 768)

        # Upsamples to pixel-level logits
        return self.seg_head(fused)              # (B, 20, 128, 128)

    def trainable_params(self):
        return [
            {'params': (p for p in self.ms_encoder.parameters() if p.requires_grad), 'name': 'ms_encoder'},
            {'params': (p for p in self.sar_encoder.parameters() if p.requires_grad), 'name': 'sar_encoder'},
            {'params': (p for p in self.fusion.parameters() if p.requires_grad), 'name': 'fusion'},
            {'params': (p for p in self.seg_head.parameters() if p.requires_grad), 'name': 'seg_head'}
        ]

    def param_counts(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'trainable': trainable, 'frozen': total - trainable}


# Instantiates model 
model = CROMAXFusionSegModel(
    croma_ckpt      = CROMA_CKPT,
    saratr_ckpt     = SARATR_CKPT,
    num_classes     = NUM_CLASSES,
    img_size        = IMG_SIZE,
    in_sar_channels_total = N_DATES_TO_USE * SAR_CHANNELS, 
    n_dates         = N_DATES_TO_USE,
    unfreeze_encoder_blocks=0 
).to(DEVICE)

counts = model.param_counts()
print(f"Total    : {counts['total']:,}")
print(f"Trainable: {counts['trainable']:,}  <- fusion + seg head + unfrozen encoder blocks")
print(f"Frozen   : {counts['frozen']:,}  <- CROMA + SARATR-X (partially frozen)")

# Full end-to-end forward pass from real data
with torch.no_grad():
    s2_b, sar_b, lbl_b = next(iter(train_loader))
    logits = model(s2_b.to(DEVICE), sar_b.to(DEVICE))
    print(f'\nEnd-to-end forward: {s2_b.shape} + {sar_b.shape} -> {logits.shape}')
    assert logits.shape == (BATCH_SIZE, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
    print('Full forward pass PASSED')

Initializing optical encoder
CROMA Optical Encoder initialized. Parameters frozen/unfrozen as configured.
SAR encoder: patch_size=8 → N_patches=256
timm ViT fallback created. N_patches=256
SARATR-X SAR Processor initialized. Parameters frozen/unfrozen as configured.
Total    : 206,680,248
Trainable: 30,344,376  <- fusion + seg head + unfrozen encoder blocks
Frozen   : 176,335,872  <- CROMA + SARATR-X (partially frozen)

End-to-end forward: torch.Size([1, 400, 128, 128]) + torch.Size([1, 120, 128, 128]) -> torch.Size([1, 20, 128, 128])
Full forward pass PASSED


In [ ]:
import time
import sys
import numpy as np
import torch
import os

import torch.nn as nn
import torch.nn.functional as F

class PanopticMultiTaskLoss(nn.Module):
    def __init__(self, ignore_index=19):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.ignore_index = ignore_index

    def dice_loss(self, inputs, targets):
        smooth = 1e-6
        num_classes = inputs.shape[1]
        inputs = F.softmax(inputs, dim=1)
        valid_mask = (targets != self.ignore_index).float()
        total_dice_loss = 0.0
        num_valid_classes = 0

        for cls in range(num_classes):
            if cls == self.ignore_index:
                continue
            input_cls_masked = inputs[:, cls, :, :] * valid_mask
            target_cls_masked = (targets == cls).float() * valid_mask
            intersection = (input_cls_masked * target_cls_masked).sum()
            union = (input_cls_masked + target_cls_masked).sum()
            dice = (2. * intersection + smooth) / (union + smooth)
            total_dice_loss += (1 - dice)
            num_valid_classes += 1

        if num_valid_classes == 0:
            return torch.tensor(0.0, device=inputs.device)
        return total_dice_loss / num_valid_classes

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        d_loss = self.dice_loss(logits, targets)
        return ce_loss + 1.0 * d_loss

# --- 1. Metric Function ---
def compute_miou(preds, targets, num_classes=NUM_CLASSES, ignore_idx=IGNORE_INDEX):
    if preds.dim() == 4:
        preds = preds.argmax(dim=1)
    valid = targets != ignore_idx
    ious  = []
    for cls in range(num_classes):
        if cls == ignore_idx: continue
        p = (preds   == cls) & valid
        t = (targets == cls) & valid
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return float(np.mean(ious)) if ious else 0.0

# --- 2. Setup & Initialization ---
CHECKPOINT_DIR = '/kaggle/working/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RESUME_FROM_CHECKPOINT = None # '/content/drive/MyDrive/Major_Project/outputs/checkpoints/latest_checkpoint.pt' # Example: set this to resume

best_miu = 0.0
history  = {'train_loss': [], 'val_loss': [], 'val_miou': []}

patience = 15
trigger_times = 0
start_epoch = 1

criterion = PanopticMultiTaskLoss(ignore_index=IGNORE_INDEX).to(DEVICE)

model.to(DEVICE)

optimizer = torch.optim.AdamW([
    {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'ms_encoder' for p in p_group['params'] if p.requires_grad), 'lr': 5e-6, 'name': 'ms_encoder_unfrozen'},
    {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'sar_encoder' for p in p_group['params'] if p.requires_grad), 'lr': 5e-6, 'name': 'sar_encoder_unfrozen'},
    {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'fusion' for p in p_group['params'] if p.requires_grad), 'lr': 1e-4, 'name': 'fusion_module'},
    {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'seg_head' for p in p_group['params'] if p.requires_grad), 'lr': 1e-4, 'name': 'seg_head'}
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Loads checkpoint if specified
if RESUME_FROM_CHECKPOINT and os.path.exists(RESUME_FROM_CHECKPOINT):
    print(f"Resuming training from {RESUME_FROM_CHECKPOINT}...")
    checkpoint = torch.load(RESUME_FROM_CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_miu = checkpoint['best_miu']
    history = checkpoint['history'] # Load history to continue plotting
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}, best mIoU: {best_miu:.4f}")
else:
    print("Starting training from scratch.")


# Initializes AMP scaler
scaler = torch.cuda.amp.GradScaler()
torch.cuda.empty_cache() # Clear cache before starting

print('Loss, optimiser, metrics ready.')
print(f"{'='*30}\nStarting Training on {DEVICE}\n{'='*30}")

for epoch in range(start_epoch, EPOCHS + 1):
    t_epoch_start = time.time()
    model.train()

    tr_loss = 0.0
    n_batches = len(train_loader)

    for batch_idx, (s2_b, sar_b, lbl_b) in enumerate(train_loader):
        s2_b, sar_b, lbl_b = s2_b.to(DEVICE), sar_b.to(DEVICE), lbl_b.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            logits = model(s2_b, sar_b)
            loss = criterion(logits, lbl_b)

        # Scaled backward pass for AMP
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for group in optimizer.param_groups for p in group['params'] if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()

        tr_loss += loss.item()

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            sys.stdout.write(f"\rEpoch {epoch:02d} | Batch {batch_idx+1}/{n_batches} | Loss: {tr_loss/(batch_idx+1):.4f}")
            sys.stdout.flush()

    # Storing training loss for the history log
    history['train_loss'].append(tr_loss/len(train_loader))

    # Validate
    model.eval()
    val_intersections = torch.zeros(NUM_CLASSES, device=DEVICE)
    val_unions = torch.zeros(NUM_CLASSES, device=DEVICE)
    val_targets = torch.zeros(NUM_CLASSES, device=DEVICE)
    val_preds = torch.zeros(NUM_CLASSES, device=DEVICE)

    with torch.no_grad():
        for s2_b, sar_b, lbl_b in val_loader:
            s2_b, sar_b, lbl_b = s2_b.to(DEVICE), sar_b.to(DEVICE), lbl_b.to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(s2_b, sar_b)
            preds = logits.argmax(dim=1)

            valid = lbl_b != IGNORE_INDEX
            for cls in range(NUM_CLASSES):
                if cls == IGNORE_INDEX: continue
                p = (preds == cls) & valid
                t = (lbl_b == cls) & valid
                val_intersections[cls] += (p & t).sum()
                val_unions[cls] += (p | t).sum()
                val_targets[cls] += t.sum()
                val_preds[cls] += p.sum()

    per_class_iou = val_intersections / (val_unions + 1e-6)
    per_class_dice = 2 * val_intersections / (val_preds + val_targets + 1e-6)

    valid_classes = [c for c in range(NUM_CLASSES) if c != IGNORE_INDEX and val_targets[c] > 0]
    va_miou = per_class_iou[valid_classes].mean().item()
    va_mdice = per_class_dice[valid_classes].mean().item()
    history['val_miou'].append(va_miou) # Store validation mIoU

    scheduler.step()

    # --- Checkpointing & Early Stopping ---
    checkpoint_state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_miu': best_miu,
        'history': history
    }

    torch.save(checkpoint_state, os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pt'))

    if va_miou > best_miu:
        best_miu = va_miou
        trigger_times = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pt')) # Only model weights for easy deployment
        torch.save(checkpoint_state, os.path.join(CHECKPOINT_DIR, 'best_model_full_state.pt')) # Full state for best model
        improved_mark = " << BEST"
    else:
        trigger_times += 1
        improved_mark = f" (Patience: {trigger_times}/{patience})"

    print(f"\nEpoch {epoch:02d} | Loss: {tr_loss/len(train_loader):.4f} | Val mIoU: {va_miou:.4f} | Val mDice: {va_mdice:.4f} | Best: {best_miu:.4f}{improved_mark}")

    if trigger_times >= patience:
        print("Early stopping triggered!")
        break


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch # Ensure torch is imported

# Loads the best model weights
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# Initializes counters for global metric accumulation
total_intersections = torch.zeros(NUM_CLASSES, device=DEVICE)
total_unions = torch.zeros(NUM_CLASSES, device=DEVICE)
total_targets = torch.zeros(NUM_CLASSES, device=DEVICE)
total_preds = torch.zeros(NUM_CLASSES, device=DEVICE)

visualization_count = 0
MAX_VIZ_BATCHES = 2 
NUM_SAMPLES_PER_VIZ = 3 

print("Starting evaluation and visualization...")

for batch_idx, (s2_val_b, sar_val_b, lbl_val_b) in enumerate(val_loader):
    s2_val_b, sar_val_b, lbl_val_b = s2_val_b.to(DEVICE), sar_val_b.to(DEVICE), lbl_val_b.to(DEVICE)

    with torch.no_grad():
        logits = model(s2_val_b, sar_val_b)
        preds = logits.argmax(dim=1)

    # Accumulates metrics per class globally
    valid = lbl_val_b != IGNORE_INDEX
    for cls in range(NUM_CLASSES):
        if cls == IGNORE_INDEX: continue
        p = (preds == cls) & valid
        t = (lbl_val_b == cls) & valid
        
        total_intersections[cls] += (p & t).sum()
        total_unions[cls] += (p | t).sum()
        total_targets[cls] += t.sum()
        total_preds[cls] += p.sum()

    # --- Visualization  ---
    if batch_idx < MAX_VIZ_BATCHES:
        colors = plt.cm.get_cmap('tab20', NUM_CLASSES)

        plt.figure(figsize=(NUM_SAMPLES_PER_VIZ * 5, 15))

        for i in range(min(NUM_SAMPLES_PER_VIZ, s2_val_b.shape[0])): # Ensure we don't exceed batch size
            # Converts tensors to numpy for plotting
            s2_img = s2_val_b[i].cpu().numpy() # Multi-band S2
            sar_img = sar_val_b[i].cpu().numpy() # Multi-band SAR
            ground_truth = lbl_val_b[i].cpu().numpy()
            prediction = preds[i].cpu().numpy()

            # Displays S2 
            plt.subplot(4, NUM_SAMPLES_PER_VIZ, i + 1)
            # For simplicity, we can take the middle date's RGB bands for display
            mid_date_idx = (N_DATES_TO_USE // 2) * S2_CHANNELS_DATA # Start index of middle date in the concatenated S2 tensor
            rgb_s2_display = s2_img[mid_date_idx + 2:mid_date_idx + 5].transpose(1, 2, 0) # Channels last, B4,B3,B2
            rgb_s2_display = np.clip(rgb_s2_display * 0.5 + 0.5, 0, 1) # Simple denormalization and clipping
            plt.imshow(rgb_s2_display)
            plt.title(f'Batch {batch_idx+1} Sample {i+1}\nS2 Input')
            plt.axis('off')

            # Displays SAR 
            plt.subplot(4, NUM_SAMPLES_PER_VIZ, i + NUM_SAMPLES_PER_VIZ + 1)
            mid_date_sar_idx = (N_DATES_TO_USE // 2) * SAR_CHANNELS # Start index of middle date in concatenated SAR tensor
            sar_vv_display = sar_img[mid_date_sar_idx] # Shape (H, W)
            sar_vv_display = (sar_vv_display - sar_vv_display.min()) / (sar_vv_display.max() - sar_vv_display.min() + 1e-6)
            plt.imshow(sar_vv_display, cmap='gray')
            plt.title(f'Batch {batch_idx+1} Sample {i+1}\nSAR Input (VV)')
            plt.axis('off')

            # Displays Ground Truth
            plt.subplot(4, NUM_SAMPLES_PER_VIZ, i + (2 * NUM_SAMPLES_PER_VIZ) + 1)
            plt.imshow(ground_truth, cmap=colors, vmin=0, vmax=NUM_CLASSES-1)
            plt.title(f'Batch {batch_idx+1} Sample {i+1}\nGround Truth')
            plt.axis('off')

            # Displays Prediction
            plt.subplot(4, NUM_SAMPLES_PER_VIZ, i + (3 * NUM_SAMPLES_PER_VIZ) + 1)
            plt.imshow(prediction, cmap=colors, vmin=0, vmax=NUM_CLASSES-1)
            plt.title(f'Batch {batch_idx+1} Sample {i+1}\nPrediction')
            plt.axis('off')

        plt.tight_layout()
        filename = f'prediction_visualization_batch_{batch_idx+1}.png'
        plt.savefig(filename)
        print(f"Saved visualization for batch {batch_idx+1} to {filename}")
        plt.close() # Close the figure to free up memory
    
    if (batch_idx + 1) % 10 == 0:
        print(f"Processed {batch_idx+1}/{len(val_loader)} validation batches...")

# Calculates final metrics
per_class_iou = total_intersections / (total_unions + 1e-6)
per_class_dice = 2 * total_intersections / (total_preds + total_targets + 1e-6)

print("\n--- Per-Crop Evaluation Metrics ---")
valid_classes = []
for cls in range(NUM_CLASSES):
    if cls == IGNORE_INDEX: continue
    if total_targets[cls] > 0: # Only compute for classes that exist in the validation set
        iou = per_class_iou[cls].item()
        dice = per_class_dice[cls].item()
        valid_classes.append(cls)
        print(f"Class {cls:02d}: IoU = {iou:.4f}, Dice = {dice:.4f}")

final_miou = per_class_iou[valid_classes].mean().item()
final_mdice = per_class_dice[valid_classes].mean().item()

print(f"\nEvaluation complete.")
print(f"Overall mIoU on validation set:  {final_miou:.4f}")
print(f"Overall mDice on validation set: {final_mdice:.4f}")